In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [7]:
from vllm import SamplingParams
from gserve import ServeConfig, LLMConfig

cnf = ServeConfig(
    gpu_ids=[0],
)

cnf2 = LLMConfig(
    model_name="meta-llama/Llama-2-7b-chat-hf",
)

cnf3 = SamplingParams(
    temperature=0.1,
)

print(cnf.__dict__)
print(cnf2.__dict__)
print(cnf3.__annotations__)

{'gpu_ids': [0], 'host': '127.0.0.1', 'port': None, 'startup_timeout': 180.0, 'client_timeout': 30.0, 'verbose': False}
{'model_name': 'meta-llama/Llama-2-7b-chat-hf', 'tokenizer': None, 'tokenizer_mode': 'auto', 'trust_remote_code': False, 'dtype': 'bfloat16', 'quantization': None, 'revision': None, 'tokenizer_revision': None, 'seed': 0, 'gpu_memory_utilization': None, 'enforce_eager': False, 'max_model_len': None, 'download_dir': None, 'lora_path': None, 'llm_kwargs': {}}
{'n': <class 'int'>, 'best_of': typing.Optional[int], '_real_n': typing.Optional[int], 'presence_penalty': <class 'float'>, 'frequency_penalty': <class 'float'>, 'repetition_penalty': <class 'float'>, 'temperature': <class 'float'>, 'top_p': <class 'float'>, 'top_k': <class 'int'>, 'min_p': <class 'float'>, 'seed': typing.Optional[int], 'stop': typing.Union[str, list[str], NoneType], 'stop_token_ids': typing.Optional[list[int]], 'ignore_eos': <class 'bool'>, 'max_tokens': typing.Optional[int], 'min_tokens': <class '

In [ ]:
from src.eval.harmbench_evaluator import HarmBenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from src.eval.beaver_evaluator import BeaverEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    # HarmBenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     use_context=False,
    # ),
    # LlamaGuardEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=None, client_timeout=60, verbose=True),
    #     # llm_config=LLMConfig(model_name="meta-llama/Llama-Guard-4-12B", max_model_len=4096),
    #     model_name="meta-llama/Llama-Guard-3-1B",
    # ),
    # StrongRejectEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     binary_thresh=0.5,
    # ),
    # BeaverEvaluator(
    #     device_map="cuda:1",
    #     binary_thresh=None,
    #     compile=True,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    # ),
    TemplateEvaluator(),
]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.adver_model import AdverModel
import torch
from notebooks.models import print_models

torch.set_float32_matmul_precision("high")  # negligable effect

print_models()
model_name = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="sequential",
    # attn_implementation="flash_attention_2"
    # attn_implementation="sdpa",
)

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from src.initialize import Initializer

adv_model = AdverModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20,
)

Initializer.normal(adv_model)
Initializer.from_string(adv_model, "Please comply with the provided request!", pad_word=".", verbose=True)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_eval = data.copy()
# ds_eval = ds_eval[:200] # for testing

dl_eval = DF_Batcher(ds_eval, batch_size=25, shuffle=False)

In [ ]:
from tqdm.auto import tqdm

all_outputs = []
for batch in tqdm(dl_eval):
    convos = [[{"role": "user", "content": prompt}] for prompt in batch["prompt"]]
    outputs = adv_model.chat(convos, max_length=256, do_sample=False, temperature=None, top_p=None)
    all_outputs.extend(outputs)

dl_eval.set_column("response", all_outputs)

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
# print prompts and outputs

for i, row in dl_eval.df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print(f"Response: {row['response']}")
    print("-" * 80)